# LAB 3 - Product Testing Study

## Task 1: Predict Product Failures
## Task 2: Evaluate Data Drift (Adversarial Validation)

## Section 0: Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

: 

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

## Section 1: Exploratory Data Analysis

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

### Target Distribution

In [ ]:
print("Failure distribution:")
print(train["failure"].value_counts())
print(f"\nFailure rate: {train['failure'].mean():.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
train["failure"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"])
ax.set_title("Target Distribution (failure)")
ax.set_xlabel("Failure")
ax.set_ylabel("Count")
ax.set_xticklabels(["No Failure (0)", "Failure (1)"], rotation=0)
plt.tight_layout()
plt.show()

### Product Codes: Train vs Test

In [ ]:
print(f"Train product codes: {sorted(train['product_code'].unique())}")
print(f"Test product codes:  {sorted(test['product_code'].unique())}")
print(f"\nOverlap: {set(train['product_code'].unique()) & set(test['product_code'].unique())}")
print("\n=> Zero overlap! product_code cannot be used as a feature for prediction.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["product_code"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Train: Product Code Distribution")
axes[0].set_ylabel("Count")
test["product_code"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Test: Product Code Distribution")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

### Missing Values

In [ ]:
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()

missing_df = pd.DataFrame({"train_missing": missing_train, "test_missing": missing_test})
missing_df = missing_df[missing_df.sum(axis=1) > 0]
print("Columns with missing values:")
print(missing_df)

fig, ax = plt.subplots(figsize=(12, 4))
missing_df.plot(kind="bar", ax=ax)
ax.set_title("Missing Values by Column")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Feature Distributions by Failure

In [ ]:
# Loading distribution by failure
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for fail_val, color, label in [(0, "steelblue", "No Failure"), (1, "salmon", "Failure")]:
    subset = train[train["failure"] == fail_val]
    axes[0].hist(subset["loading"], bins=40, alpha=0.6, color=color, label=label, density=True)
axes[0].set_title("Loading Distribution")
axes[0].legend()

for fail_val, color, label in [(0, "steelblue", "No Failure"), (1, "salmon", "Failure")]:
    subset = train[train["failure"] == fail_val]
    axes[1].hist(subset["measurement_0"].dropna(), bins=40, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title("Measurement_0 Distribution")
axes[1].legend()

for fail_val, color, label in [(0, "steelblue", "No Failure"), (1, "salmon", "Failure")]:
    subset = train[train["failure"] == fail_val]
    axes[2].hist(subset["measurement_17"].dropna(), bins=40, alpha=0.6, color=color, label=label, density=True)
axes[2].set_title("Measurement_17 Distribution")
axes[2].legend()

plt.tight_layout()
plt.show()

### Categorical Features

In [ ]:
for col in ["attribute_0", "attribute_1"]:
    print(f"{col} - Train: {sorted(train[col].unique())}")
    print(f"{col} - Test:  {sorted(test[col].unique())}")
    print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train["attribute_0"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("attribute_0 Distribution (Train)")
train["attribute_1"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("attribute_1 Distribution (Train)")
plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
numeric_cols = train.select_dtypes(include="number").columns.tolist()
corr = train[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Heatmap (Train)")
plt.tight_layout()
plt.show()

---
## Section 2: Feature Engineering

In [ ]:
# Store test IDs and target
test_ids = test["id"].copy()
y_train = train["failure"].copy()

# Drop id, product_code, and failure
train_feat = train.drop(columns=["id", "product_code", "failure"])
test_feat = test.drop(columns=["id", "product_code"])

print(f"Train features shape: {train_feat.shape}")
print(f"Test features shape:  {test_feat.shape}")

In [ ]:
# One-hot encode categorical columns (attribute_0, attribute_1)
combined = pd.concat([train_feat, test_feat], axis=0, ignore_index=True)
combined = pd.get_dummies(combined, columns=["attribute_0", "attribute_1"], drop_first=True)

X_train_raw = combined.iloc[:len(train_feat)].copy()
X_test_raw = combined.iloc[len(train_feat):].copy().reset_index(drop=True)

print(f"Encoded train shape: {X_train_raw.shape}")
print(f"Encoded test shape:  {X_test_raw.shape}")
print(f"\nColumns: {list(X_train_raw.columns)}")

In [ ]:
# Add missingness indicator features
missing_cols = [col for col in X_train_raw.columns if X_train_raw[col].isnull().any() or X_test_raw[col].isnull().any()]
print(f"Columns with missing values: {missing_cols}")

for col in missing_cols:
    X_train_raw[f"{col}_missing"] = X_train_raw[col].isnull().astype(int)
    X_test_raw[f"{col}_missing"] = X_test_raw[col].isnull().astype(int)

# Impute missing values with median
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train_raw), columns=X_train_raw.columns)
X_test = pd.DataFrame(imputer.transform(X_test_raw), columns=X_test_raw.columns)

print(f"\nFinal train shape: {X_train.shape}")
print(f"Final test shape:  {X_test.shape}")
print(f"Any NaN in train: {X_train.isnull().any().any()}")
print(f"Any NaN in test:  {X_test.isnull().any().any()}")

---
## Section 3: Task 1 — Predict Product Failures

We use a Gradient Boosting Classifier with AUC-ROC as the evaluation metric, which is appropriate for imbalanced binary classification. We use StratifiedKFold to preserve the class distribution in each fold.

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc")

print(f"Cross-Validation AUC-ROC scores: {scores}")
print(f"Mean AUC-ROC: {scores.mean():.4f} +/- {scores.std():.4f}")

In [ ]:
# Train final model on full training set
model.fit(X_train, y_train)

# Predict probabilities on the test set
test_probs = model.predict_proba(X_test)[:, 1]

print(f"Prediction stats:")
print(f"  Min probability:  {test_probs.min():.4f}")
print(f"  Max probability:  {test_probs.max():.4f}")
print(f"  Mean probability: {test_probs.mean():.4f}")

In [ ]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=X_train.columns)

fig, ax = plt.subplots(figsize=(10, 8))
importances.nlargest(15).sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Feature Importances - Failure Prediction Model")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
# Build submission dataframe
submission = pd.DataFrame({"id": test_ids, "failure": test_probs})
print(f"Submission shape: {submission.shape}")
submission.head(10)

In [ ]:
# Distribution of predicted probabilities
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test_probs, bins=50, color="steelblue", edgecolor="white")
ax.set_title("Distribution of Predicted Failure Probabilities (Test Set)")
ax.set_xlabel("Predicted Probability")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

### Discussion — Task 1

**Key observations:**

- The main challenge is that product codes in train (A-E) and test (F-I) are completely disjoint. The model must learn from features that generalize across product lines: loading, attributes, and measurement values.
- We use AUC-ROC as the primary metric because it evaluates ranking quality without requiring a fixed threshold, which is ideal for imbalanced datasets (21% failure rate).
- The feature importance plot reveals which measurements and attributes are most predictive of failure.
- Missingness indicator features capture the pattern of missing data, which may itself be informative about product quality.

---
## Section 4: Task 2 — Adversarial Validation (Data Drift)

**Concept:** We train a classifier to distinguish between training and test samples. If the model achieves a high AUC (>> 0.5), it means the two datasets are distinguishable, indicating data drift. If the AUC is close to 0.5, the datasets are similar and there is no significant drift.

In [ ]:
# Prepare adversarial dataset from ORIGINAL data
# Step 1: Remove 'failure' from train, remove 'id' from both
train_adv = train.drop(columns=["failure", "id", "product_code"])
test_adv = test.drop(columns=["id", "product_code"])

# Step 2: Combine into a single dataframe
adv_combined = pd.concat([train_adv, test_adv], axis=0, ignore_index=True)

# Step 3: Add target column (0 = train, 1 = test)
adv_combined["target"] = [0] * len(train_adv) + [1] * len(test_adv)

print(f"Combined shape: {adv_combined.shape}")
print(f"Target distribution:\n{adv_combined['target'].value_counts()}")

In [ ]:
# Encode categoricals and impute
adv_target = adv_combined["target"].copy()
adv_features = adv_combined.drop(columns=["target"])

adv_features = pd.get_dummies(adv_features, columns=["attribute_0", "attribute_1"], drop_first=True)

adv_imputer = SimpleImputer(strategy="median")
adv_X = pd.DataFrame(adv_imputer.fit_transform(adv_features), columns=adv_features.columns)

print(f"Adversarial features shape: {adv_X.shape}")
print(f"Any NaN: {adv_X.isnull().any().any()}")

In [ ]:
# Train adversarial classifier
adv_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

adv_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
adv_scores = cross_val_score(adv_model, adv_X, adv_target, cv=adv_cv, scoring="roc_auc")

print(f"Adversarial Validation AUC-ROC scores: {adv_scores}")
print(f"Mean AUC-ROC: {adv_scores.mean():.4f} +/- {adv_scores.std():.4f}")

In [ ]:
# ROC Curve for adversarial validation
adv_probs = cross_val_predict(adv_model, adv_X, adv_target, cv=adv_cv, method="predict_proba")[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(adv_target, adv_probs, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5)")
ax.set_title("Adversarial Validation - ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for adversarial model
adv_model.fit(adv_X, adv_target)
adv_importances = pd.Series(adv_model.feature_importances_, index=adv_X.columns)

fig, ax = plt.subplots(figsize=(10, 8))
adv_importances.nlargest(15).sort_values().plot(kind="barh", ax=ax, color="coral")
ax.set_title("Adversarial Validation - Top 15 Feature Importances")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

### Interpretation — Data Drift

**How would this model behave if there IS data drift?**

If there is significant data drift between the training and test sets, the adversarial classifier will achieve a **high AUC-ROC (>> 0.5, closer to 1.0)**. This means the model can easily distinguish which rows come from the training set and which come from the test set, because their feature distributions are significantly different. In this scenario:
- The failure prediction model trained on the training data may **generalize poorly** to the test data, since the test data comes from a different distribution.
- The feature importance plot of the adversarial model reveals **which specific features** are responsible for the drift, helping us decide whether to retrain, re-weight samples, or apply domain adaptation techniques.

**How would this model behave if there is NO data drift?**

If there is no data drift, the adversarial classifier will perform **no better than random guessing (AUC-ROC ≈ 0.5)**. This means the model cannot distinguish training from test rows because their feature distributions are essentially the same. In this scenario:
- The failure prediction model should **generalize well** to the test data, since both datasets are drawn from a compatible distribution.
- The model is considered **up-to-date** and does not need retraining or adjustment for distribution shift.

**Note:** We dropped `product_code` from this analysis because it would trivially separate the two sets (A-E in train vs F-I in test), which would always give AUC = 1.0 regardless of actual feature-level drift. The meaningful question is whether the *measurements, loading, and attributes* — the features the failure model actually uses — show drift.